# Logistic Regression — Strategy B: Random Undersampling

Logistic Regression is trained using Strategy B, where random undersampling is applied to reduce the majority non-diabetes class in the training data. This creates a more balanced training set without generating synthetic samples. Undersampling is included inside the pipeline during cross-validation so that only the training folds are resampled, while validation and test data remain unchanged.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
import shap
import numpy as np

In [20]:
X_train_final = pd.read_parquet("../DATASETS/PREPROCESSED/X_train_final.parquet")
X_test_final = pd.read_parquet("../DATASETS/PREPROCESSED/X_test_final.parquet")

y_train = pd.read_parquet("../DATASETS/PREPROCESSED/y_train.parquet")["diabetes"]
y_test = pd.read_parquet("../DATASETS/PREPROCESSED/y_test.parquet")["diabetes"]

In [21]:
pipeline = Pipeline([
    ("undersampling", RandomUnderSampler(random_state=42)),
    ("model", LogisticRegression(max_iter=5000, random_state=42))
])

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear"]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_final, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV ROC AUC:", grid_search.best_score_)

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test_final)
y_pred_proba = best_model.predict_proba(X_test_final)[:, 1]

Fitting 10 folds for each of 12 candidates, totalling 120 fits
Best parameters: {'model__C': 0.1, 'model__penalty': 'l2', 'model__solver': 'liblinear'}
Best CV ROC AUC: 0.8157524421554342


## Final Test Set Evaluation

After hyperparameter tuning, the best model is evaluated on the held-out test set. The test set is not resampled and therefore preserves the original class distribution. Performance is assessed using accuracy, precision, recall, F1-score, ROC AUC, and the confusion matrix.

In [22]:
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
print("Confusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7168
ROC AUC: 0.8101
Confusion Matrix:
[[2995 1224]
 [ 249  733]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.71      0.80      4219
         1.0       0.37      0.75      0.50       982

    accuracy                           0.72      5201
   macro avg       0.65      0.73      0.65      5201
weighted avg       0.82      0.72      0.75      5201



## Exporting Model Results

The final evaluation metrics are exported to a CSV file so they can be combined later in a separate model comparison notebook. This avoids retraining models when creating summary tables and visualizations.

In [ ]:
metrics = {

    "Strategy": "Undersampling",

    "Model": "Logistic Regression",

    "Accuracy": accuracy_score(y_test, y_pred),

    "Precision": precision_score(y_test, y_pred),

    "Recall": recall_score(y_test, y_pred),

    "F1-score": f1_score(y_test, y_pred),

    "ROC AUC": roc_auc_score(y_test, y_pred_proba)

}

metrics_df = pd.DataFrame([metrics])

metrics_df.to_csv("../RESULTS/PERFORMANCE/logistic_regression_undersampling_metrics.csv", index=False)

In [ ]:
import numpy as np

lr_model = best_model.named_steps["model"]

explainer = shap.LinearExplainer(
    lr_model,
    X_train_final,
    feature_perturbation="interventional"
)

shap_values = explainer.shap_values(X_test_final)

shap_importance = np.mean(np.abs(shap_values), axis=0)

shap_results = pd.DataFrame({
    "model": "Logistic Regression",
    "strategy": "Undersampling",
    "method": "SHAP",
    "feature": X_test_final.columns,
    "importance": shap_importance
})

shap_results = shap_results.sort_values("importance", ascending=False).reset_index(drop=True)
shap_results["rank"] = (shap_results["importance"]
                         .rank(method="first", ascending=False)
                         .astype(int))

shap_results = shap_results[["model", "strategy", "method", "feature", "importance", "rank"]]
shap_results.to_csv("../RESULTS/PERFORMANCE/logistic_regression_metrics.csv", index=False)



c:\Users\itchy\anaconda3\Lib\site-packages\shap\explainers\_linear.py:123: FutureWarning: The feature_perturbation option is now deprecated in favor of using the appropriate masker (maskers.Independent, maskers.Partition or maskers.Impute).
  warnings.warn(wmsg, FutureWarning)


In [ ]:
from sklearn.inspection import permutation_importance

lr_model = best_model.named_steps["model"]

perm_result = permutation_importance(
    lr_model,
    X_test_final,
    y_test,
    n_repeats=30,
    random_state=42,
    scoring="roc_auc",
    n_jobs=-1
)

perm_importance = perm_result.importances_mean

perm_results = pd.DataFrame({
    "model": "Logistic Regression",
    "strategy": "Undersampling",
    "method": "Permutation",
    "feature": X_test_final.columns,
    "importance": perm_importance
})

perm_results = perm_results.sort_values("importance", ascending=False).reset_index(drop=True)
perm_results["rank"] = (
    perm_results["importance"]
    .rank(method="first", ascending=False)
    .astype(int)
)

perm_results = perm_results[["model", "strategy", "method", "feature", "importance", "rank"]]
perm_results.to_csv("../RESULTS/PERFORMANCE/logistic_regression_metrics.csv", index=False)

In [33]:
lr_model = best_model.named_steps["model"]

coef_importance = np.abs(lr_model.coef_[0])

coef_results = pd.DataFrame({
    "model": "Logistic Regression",
    "strategy": "Undersampling",
    "method": "Coefficient",
    "feature": X_test_final.columns,
    "importance": coef_importance
})

coef_results = coef_results.sort_values("importance", ascending=False).reset_index(drop=True)
coef_results["rank"] = (
    coef_results["importance"]
    .rank(method="first", ascending=False)
    .astype(int)
)

coef_results = coef_results[["model", "strategy", "method", "feature", "importance", "rank"]]
coef_results.to_csv("../RESULTS/PERFORMANCE/logistic_regression_metrics.csv", index=False)